# Lab Day 10 trên Kaggle: QLoRA Qwen3-VL-4B

Chạy E0, E1, E2, E3 và E4 (10% / 100% dữ liệu). Notebook nhúng toàn bộ mã cần thiết.
1. Chọn GPU **T4** (nếu T4 x2, mã dùng GPU 0), bật **Internet**.
2. Add Input: dataset `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`.
3. Chọn **Save Version → Save & Run All**. Không phụ thuộc biến của phiên interactive.
4. Tải `/kaggle/working/lab_day10_results.zip` trong Output để gửi phân tích.

E5/GRPO là bonus, chưa có trong notebook này. Số liệu không được tạo giả; báo cáo được viết sau khi chạy.
Nếu yêu cầu lớp học bắt buộc Colab, xác nhận với giảng viên việc dùng Kaggle.


## 0. Cấu hình lượt chạy

In [1]:
# Mặc định chạy đầy đủ 300 bước cho mỗi lượt train.
# Có thể đặt MAX_STEPS=2 để kiểm tra pipeline; lượt này không dùng làm bài nộp.
DATA_SOURCE = None  # tự tìm trong /kaggle/input; hoặc đường dẫn root có images/ và labels-YOLO/
RESUME_SOURCE = None  # root day10 của Output cũ đã Add Input; chỉ tái dùng lượt đã hoàn thành
MODEL_REVISION = None  # tự ghim SHA model; khi resume, dùng SHA từ kaggle_run_config.json cũ
MAX_STEPS = 300
BATCH_SIZE = 1
GRAD_ACCUM = 8
INFERENCE_BATCH_SIZE = 2
MAX_NEW_TOKENS = 512  # dùng cùng giới hạn cho mọi setup, giảm nguy cơ cắt JSON nhiều box


## 1. Khôi phục mã nguồn vào working

In [2]:
import base64, io, json, os, subprocess, sys, zipfile
from pathlib import Path

BASE = Path("/kaggle/working")
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / "day10"
PROJECT.mkdir(exist_ok=True)
PAYLOAD = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(PROJECT)
os.environ["HF_HOME"] = "/tmp/day10_hf_cache"  # model gốc không nằm trong Output
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
config = {"input_root": "/kaggle/input", "data_source": DATA_SOURCE,
          "resume_source": RESUME_SOURCE, "model_revision": MODEL_REVISION,
          "max_steps": MAX_STEPS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
          "inference_batch_size": INFERENCE_BATCH_SIZE, "max_new_tokens": MAX_NEW_TOKENS}
if BATCH_SIZE * GRAD_ACCUM != 8:
    raise ValueError("Giữ effective batch size = 8 cho các thí nghiệm.")
if not all(isinstance(v, int) and v > 0 for v in
           [MAX_STEPS, BATCH_SIZE, GRAD_ACCUM, INFERENCE_BATCH_SIZE, MAX_NEW_TOKENS]):
    raise ValueError("Các cấu hình bước/batch/token phải là số nguyên dương.")
(PROJECT / "kaggle_run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Project:", PROJECT)


Project: /kaggle/working/day10


## 2. Cài thư viện, kiểm tra GPU và công cụ đề

In [3]:
# Cài trước khi kernel import torch/transformers. Giữ bản CUDA torch/vision của Kaggle.
subprocess.run(["nvidia-smi", "-L"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "import torch, torchvision; assert torch.cuda.is_available(), 'GPU required'; "
    "print('torch',torch.__version__,'torchvision',torchvision.__version__); "
    "assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,4), 'torch>=2.4 required'"],
    check=True)
import importlib.metadata as package_metadata
constraints = PROJECT / "kaggle_torch_constraints.txt"
versions = []
for name in ("torch", "torchvision", "torchaudio"):
    try:
        versions.append(name + "==" + package_metadata.version(name))
    except package_metadata.PackageNotFoundError:
        pass
constraints.write_text(chr(10).join(versions) + chr(10), encoding="utf-8")
print("Keeping Kaggle CUDA packages:", versions)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT / "requirements-kaggle.txt"), "-c", str(constraints)], check=True)
# This pipeline uses bitsandbytes NF4. A preinstalled TorchAO can break PEFT's
# optional backend detection (PEFT 0.21.2 rejects torchao < 0.16.0).
try:
    torchao_version = package_metadata.version("torchao")
except package_metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    print("Removing unused optional TorchAO backend:", torchao_version)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "kaggle_checks"],
               cwd=PROJECT, check=True)


GPU 0: Tesla T4 (UUID: GPU-8d1860b4-83cd-0aba-0cae-3990e23477c0)
GPU 1: Tesla T4 (UUID: GPU-4670442a-40ab-2195-7a66-b97bc605852c)
torch 2.11.0+cu128 torchvision 0.26.0+cu128
Keeping Kaggle CUDA packages: ['torch==2.11.0+cu128', 'torchvision==0.26.0+cu128', 'torchaudio==2.11.0+cu128']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', 'tests', 'kaggle_checks'], returncode=0)

## 3. Run All: dữ liệu → E0–E3 → chọn bằng val → E4 → xuất kết quả

In [4]:
# Các setup chạy ở subprocess riêng, giải phóng GPU giữa các thí nghiệm.
# subprocess.check_call truyền log trực tiếp vào Output của Save Version.
try:
    subprocess.run([sys.executable, "-u", "kaggle_lab.py", "--task", "all"],
                   cwd=PROJECT, check=True)
except subprocess.CalledProcessError:
    print("Pipeline dừng vì lỗi. Xem day10/report/failure.json và log bên trên.")
    print("ZIP chứa kết quả đã hoàn thành vẫn được tạo nếu runner đã khởi động.")
    raise


dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}

Fetching 2 files: 100%|██████████| 2/2 [00:44<00:00, 22.08s/it]

Loading weights: 100%|██████████| 713/713 [00:08<00:00, 88.34it/s] 
NF4 layers: 252; visual and lm_head excluded from quantization
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[tr

## 4. Xem kết quả và tải file

In [5]:
import csv
from IPython.display import FileLink, display
summary = PROJECT / "report/summary.csv"
if summary.exists():
    import pandas as pd
    display(pd.read_csv(summary))
quality = PROJECT / "report/quality_checks.json"
if quality.exists():
    print(quality.read_text(encoding="utf-8"))
display(FileLink("lab_day10_results.zip"))
print("Checkpoint: /kaggle/working/day10/work/<experiment>/adapter/")
print("Projector E2/E3: /kaggle/working/day10/work/<experiment>/projector.pt")
print("Gửi ZIP kết quả để phân tích và viết REPORT.md. Không commit trọng số/dữ liệu.")


,experiment,split,f1,ci95_low,ci95_high,fp_per_image,parse_valid_rate,missing_predictions,f1_pothole,f1_crack,f1_manhole,trainable_params,peak_vram_gb,train_minutes
0,e0,test,0.077470,0.056632,0.099467,2.035370,0.909968,0,0.153846,0.023107,0.169697,0,3.621544,0.000000
1,e1,test,0.278719,0.240359,0.319617,1.064309,0.993569,0,0.180851,0.202899,0.537549,33030144,5.064273,51.742330
2,e2,test,0.353716,0.313796,0.394712,0.938907,1.000000,0,0.201923,0.257496,0.652778,142135296,6.725460,52.679198
3,e3,test,0.433206,0.388634,0.476008,0.765273,1.000000,0,0.290155,0.337979,0.725979,148426752,6.860916,62.296972
4,e4_data_size_10,test,0.283321,0.249812,0.317315,1.749196,1.000000,0,0.191882,0.226316,0.524823,148426752,6.588425,61.631699
5,e4_data_size_100,test,0.433206,0.388634,0.476008,0.765273,1.000000,0,0.290155,0.337979,0.725979,148426752,6.860916,62.296972


{
  "e1_parse_valid_at_least_095": true,
  "e1_f1_gain_at_least_010": true,
  "e1_f1_gain": 0.20124904163223983,
  "all_test_predictions_present": true,
  "status": "finished",
  "report_and_git_checks_pending": true,
  "smoke_run": false
}


/kaggle/working/lab_day10_results.zip

Checkpoint: /kaggle/working/day10/work/<experiment>/adapter/
Projector E2/E3: /kaggle/working/day10/work/<experiment>/projector.pt
Gửi ZIP kết quả để phân tích và viết REPORT.md. Không commit trọng số/dữ liệu.
